# End-to-End Audio Deepfake Detection Pipeline
### Pretrained Wav2Vec 2.0 Acoustic Classifier

This interactive notebook guides you through the full workflow:
- **Step 0:** Virtual Environment & Dependency Setup
- **Step 1:** Data Exploration & Inspection
- **Step 2:** Audio Cleaning & Preprocessing (16kHz Mono, 4.0s Equalization, Peak Normalization)
- **Step 3:** Exploratory Data Analysis & Visualizations
- **Step 4:** Model Architecture & Mixed Precision Training
- **Step 5:** Model Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC)
- **Step 6:** Model Export & Real-Time Inference

In [ ]:
# Step 0: Imports and Device Check
import os, sys, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
import librosa, librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

BASE_DIR = Path.cwd()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute Device: {device} ({torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'})")


In [ ]:
# Step 1: Summary of Raw Dataset
summary_file = BASE_DIR / "logs" / "dataset_inspection_summary.json"
if summary_file.exists():
    with open(summary_file) as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2))
else:
    print("Run Step 1 to generate dataset inspection summary.")


In [ ]:
# Step 3: View Saved Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
img_paths = [
    BASE_DIR / "graphs" / "01_class_distribution.png",
    BASE_DIR / "graphs" / "02_duration_distribution_before_after.png",
    BASE_DIR / "graphs" / "03_waveform_comparison.png",
    BASE_DIR / "graphs" / "04_mel_spectrogram_comparison.png"
]
titles = ["Class Distribution", "Duration Distribution", "Waveform Comparison", "Mel-Spectrograms"]

for ax, p, t in zip(axes.flat, img_paths, titles):
    if p.exists():
        img = plt.imread(str(p))
        ax.imshow(img)
        ax.set_title(t, fontweight="bold")
        ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Step 5: Test Set Evaluation Metrics & Confusion Matrix
metrics_file = BASE_DIR / "evaluation" / "metrics_report.json"
if metrics_file.exists():
    with open(metrics_file) as f:
        metrics = json.load(f)
    print("="*60)
    print(f"Accuracy:  {metrics['accuracy']*100:.2f}%")
    print(f"Precision: {metrics['precision']*100:.2f}%")
    print(f"Recall:    {metrics['recall']*100:.2f}%")
    print(f"F1-Score:  {metrics['f1_score']*100:.2f}%")
    print(f"ROC-AUC:   {metrics['roc_auc']*100:.2f}%")
    print("="*60)

cm_img = BASE_DIR / "evaluation" / "confusion_matrix.png"
if cm_img.exists():
    plt.figure(figsize=(7, 6))
    plt.imshow(plt.imread(str(cm_img)))
    plt.axis("off")
    plt.title("Test Set Confusion Matrix", fontweight="bold")
    plt.show()


In [ ]:
# Step 6: Live Audio Prediction (Self-Contained)
from transformers import Wav2Vec2Model

class AudioDeepfakeDetector:
    def __init__(self, model_path=BASE_DIR / 'models' / 'best_wav2vec2_deepfake_model.pth', dev=device):
        self.device = dev
        self.bundle = torch.load(model_path, map_location=self.device, weights_only=False)
        self.target_sr = self.bundle.get('preprocessing_config', {}).get('sample_rate', 16000)
        self.target_samples = self.bundle.get('preprocessing_config', {}).get('target_samples', 64000)
        
        class Classifier(nn.Module):
            def __init__(self):
                super().__init__()
                self.wav2vec2 = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
                self.classifier = nn.Sequential(
                    nn.LayerNorm(768),
                    nn.Linear(768, 256),
                    nn.GELU(),
                    nn.Dropout(0.3),
                    nn.Linear(256, 2)
                )
            def forward(self, x):
                return torch.softmax(self.classifier(self.wav2vec2(input_values=x).last_hidden_state.mean(dim=1)), dim=-1)

        self.model = Classifier().to(self.device)
        self.model.load_state_dict(self.bundle['model_state_dict'])
        self.model.eval()

    def predict(self, audio_path):
        data, sr = sf.read(str(audio_path), dtype='float32')
        if data.ndim > 1: data = data.mean(axis=1)
        if sr != self.target_sr:
            t_data = torch.from_numpy(data).unsqueeze(0)
            data = torchaudio.transforms.Resample(sr, self.target_sr)(t_data).squeeze(0).numpy()
        if len(data) < self.target_samples: data = np.pad(data, (0, self.target_samples - len(data)))
        else: data = data[:self.target_samples]
        m = np.max(np.abs(data))
        if m > 1e-6: data = (data / m) * 0.95
        
        tensor_in = torch.from_numpy(data).unsqueeze(0).to(self.device)
        with torch.no_grad():
            probs = self.model(tensor_in).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))
        label = 'Fake' if pred_id == 1 else 'Real'
        
        return {
            'file': Path(audio_path).name,
            'prediction': label,
            'confidence': float(probs[pred_id]),
            'is_fake': bool(pred_id == 1),
            'probabilities': {'Real': float(probs[0]), 'Fake': float(probs[1])}
        }

detector = AudioDeepfakeDetector()
test_file = list((BASE_DIR / 'data' / 'processed_clean' / 'FAKE').glob('*.wav'))[0]
res = detector.predict(test_file)

print(f"File:        {res['file']}")
print(f"Prediction:  {res['prediction']} ({res['confidence']*100:.2f}% confidence)")
print(f"Is Deepfake: {res['is_fake']}")
print(f"Probabilities: {res['probabilities']}")

